In [1]:
!pip install -q sentence-transformers faiss-cpu gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.3 MB/s eta 0:00:00


In [3]:
college_data = [
    {
        "topic": "College Location",
        "information": (
            "KIET Engineering College is located in "
            "the Kakinada region of Andhra Pradesh, India."
        )
    },
    {
        "topic": "College Timings",
        "information": (
            "The illustrative college schedule is "
            "Monday to Saturday, from 9 AM to 4 PM. "
            "Confirm the actual timetable with the college."
        )
    },
    {
        "topic": "Available Branches",
        "information": (
            "Students can ask about CSE, Artificial Intelligence "
            "and Data Science, Artificial Intelligence and "
            "Machine Learning, and Cyber Security. "
            "Confirm currently offered branches with the college."
        )
    },
    {
        "topic": "Admission Documents",
        "information": (
            "Commonly requested documents include SSC marks memo, "
            "diploma or qualifying examination certificates, "
            "rank card, transfer certificate, study certificates, "
            "caste certificate when applicable, income certificate "
            "when applicable, Aadhaar, and passport-size photographs. "
            "The actual requirements depend on the admission process."
        )
    },
    {
        "topic": "Hostel Facilities",
        "information": (
            "Students should contact the college administration "
            "to confirm hostel availability, fees, accommodation, "
            "and mess facilities."
        )
    },
    {
        "topic": "Placements",
        "information": (
            "Students can contact the placement cell for verified "
            "placement statistics, recruiting companies, training, "
            "and placement opportunities."
        )
    },
    {
        "topic": "Scholarships",
        "information": (
            "Eligible students may be able to apply for scholarships "
            "or fee reimbursement schemes, depending on government "
            "rules and eligibility requirements."
        )
    }
]

print("College information loaded:", len(college_data), "topics")

College information loaded: 7 topics


In [4]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Convert college information into documents
documents = [
    f"Topic: {item['topic']}\n"
    f"Information: {item['information']}"
    for item in college_data
]

# Generate embeddings
embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings created successfully!")
print("Total documents:", len(documents))
print("Embedding shape:", embeddings.shape)

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created successfully!
Total documents: 7
Embedding shape: (7, 384)


In [5]:
import faiss
import numpy as np

# Get the embedding dimensions
dimension = embeddings.shape[1]

# Create the FAISS search index
index = faiss.IndexFlatIP(dimension)

# Convert embeddings to float32 and add them to FAISS
index.add(
    np.asarray(embeddings, dtype="float32")
)

print("FAISS search system created successfully!")
print("Total searchable documents:", index.ntotal)
print("Embedding dimensions:", dimension)

FAISS search system created successfully!
Total searchable documents: 7
Embedding dimensions: 384


In [6]:
# ============================================
# STEP 6: RETRIEVE RELEVANT COLLEGE INFORMATION
# ============================================

def retrieve_information(question, top_k=3):

    # Convert the student's question into an embedding
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Convert embedding to FAISS-compatible format
    question_embedding = np.asarray(
        question_embedding,
        dtype="float32"
    )

    # Search the FAISS index
    scores, indices = index.search(
        question_embedding,
        min(top_k, len(documents))
    )

    # Collect the retrieved information
    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        results.append({
            "topic": college_data[idx]["topic"],
            "information": college_data[idx]["information"],
            "score": float(score)
        })

    return results


# ============================================
# TEST THE RETRIEVAL FUNCTION
# ============================================

question = "What are the college timings?"

results = retrieve_information(question)

print("Student Question:", question)
print("\nRetrieved College Information:\n")

for result in results:

    print("Topic:", result["topic"])
    print("Information:", result["information"])
    print("Similarity Score:", round(result["score"], 3))
    print("-" * 50)

Student Question: What are the college timings?

Retrieved College Information:

Topic: College Timings
Information: The illustrative college schedule is Monday to Saturday, from 9 AM to 4 PM. Confirm the actual timetable with the college.
Similarity Score: 0.706
--------------------------------------------------
Topic: Placements
Information: Students can contact the placement cell for verified placement statistics, recruiting companies, training, and placement opportunities.
Similarity Score: 0.296
--------------------------------------------------
Topic: Hostel Facilities
Information: Students should contact the college administration to confirm hostel availability, fees, accommodation, and mess facilities.
Similarity Score: 0.284
--------------------------------------------------


In [7]:
test_questions = [
    "Which branches are available?",
    "What documents are required for admission?",
    "Tell me about hostel facilities.",
    "What scholarship opportunities are available?"
]

for question in test_questions:

    print("\nQuestion:", question)

    results = retrieve_information(question, top_k=1)

    if results:
        print("Retrieved Topic:", results[0]["topic"])
        print("Answer:", results[0]["information"])
    else:
        print("No relevant information found.")

    print("-" * 60)


Question: Which branches are available?
Retrieved Topic: Available Branches
Answer: Students can ask about CSE, Artificial Intelligence and Data Science, Artificial Intelligence and Machine Learning, and Cyber Security. Confirm currently offered branches with the college.
------------------------------------------------------------

Question: What documents are required for admission?
Retrieved Topic: Admission Documents
Answer: Commonly requested documents include SSC marks memo, diploma or qualifying examination certificates, rank card, transfer certificate, study certificates, caste certificate when applicable, income certificate when applicable, Aadhaar, and passport-size photographs. The actual requirements depend on the admission process.
------------------------------------------------------------

Question: Tell me about hostel facilities.
Retrieved Topic: Hostel Facilities
Answer: Students should contact the college administration to confirm hostel availability, fees, accommo

In [8]:
# ============================================
# STEP 7: GENERATE CHATBOT ANSWERS
# ============================================

def generate_answer(question):

    question = question.strip()

    # Check whether the question is empty
    if not question:
        return "Please enter a question."

    # Retrieve relevant college information
    results = retrieve_information(question, top_k=3)

    # Check whether information was retrieved
    if not results:
        return (
            "Sorry, I could not find this information "
            "in my college database."
        )

    # Select the most relevant result
    best_result = results[0]

    # Check whether the result is relevant enough
    if best_result["score"] < 0.30:
        return (
            "Sorry, I could not find relevant information "
            "in my college database. Please ask a question "
            "about college information."
        )

    # Generate the answer from the retrieved information
    answer = (
        f"Topic: {best_result['topic']}\n\n"
        f"{best_result['information']}"
    )

    return answer


# ============================================
# TEST THE CHATBOT
# ============================================

questions = [
    "What are the college timings?",
    "Which branches are available?",
    "What documents are required for admission?",
    "Tell me about scholarships.",
    "How can I cook biryani?"
]

for question in questions:

    print("Student:", question)
    print("CollegeMate AI:", generate_answer(question))
    print("-" * 60)

Student: What are the college timings?
CollegeMate AI: Topic: College Timings

The illustrative college schedule is Monday to Saturday, from 9 AM to 4 PM. Confirm the actual timetable with the college.
------------------------------------------------------------
Student: Which branches are available?
CollegeMate AI: Topic: Available Branches

Students can ask about CSE, Artificial Intelligence and Data Science, Artificial Intelligence and Machine Learning, and Cyber Security. Confirm currently offered branches with the college.
------------------------------------------------------------
Student: What documents are required for admission?
CollegeMate AI: Topic: Admission Documents

Commonly requested documents include SSC marks memo, diploma or qualifying examination certificates, rank card, transfer certificate, study certificates, caste certificate when applicable, income certificate when applicable, Aadhaar, and passport-size photographs. The actual requirements depend on the admiss

In [10]:
!pip install -q gradio

In [11]:
# ============================================
# STEP 8: COLLEGEMATE AI CHATBOT INTERFACE
# ============================================

import gradio as gr


# Chatbot response function
def chat_with_college(message, history):

    answer = generate_answer(message)

    return answer


# Create the chatbot interface
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="CollegeMate AI"
) as demo:

    gr.Markdown(
        """
        # 🎓 CollegeMate AI
        ### Your College Information Assistant

        Ask questions about college timings, branches,
        admissions, hostel facilities, placements, and scholarships.
        """
    )

    gr.Markdown("### 💡 Suggested Questions")

    with gr.Row():

        btn1 = gr.Button("🕘 College Timings")

        btn2 = gr.Button("📚 Available Branches")

        btn3 = gr.Button("📝 Admission Documents")

    chatbot = gr.ChatInterface(
        fn=chat_with_college,
        chatbot=gr.Chatbot(
            height=400
        ),
        textbox=gr.Textbox(
            placeholder="Ask your college question here...",
            label="Your Question"
        ),
        title="Chat with CollegeMate AI",
        description="Enter a question or select a suggested question."
    )

    # Connect suggested question buttons
    btn1.click(
        fn=lambda: "What are the college timings?",
        outputs=chatbot.textbox
    )

    btn2.click(
        fn=lambda: "Which branches are available?",
        outputs=chatbot.textbox
    )

    btn3.click(
        fn=lambda: "What documents are required for admission?",
        outputs=chatbot.textbox
    )


# Launch the application
demo.launch(share=True)

/tmp/ipykernel_528/3959352852.py:17: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bf5755337a940d39f2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
# ============================================
# STEP 9: IMPROVE COLLEGEMATE AI
# ============================================

def generate_answer(question):

    question = question.strip()

    # 1. Check for empty input
    if not question:
        return "Please enter a question."

    # 2. Define college-related keywords
    college_keywords = [
        "college", "kiet", "branch", "branches",
        "admission", "document", "documents",
        "hostel", "placement", "placements",
        "scholarship", "scholarships", "fee", "fees",
        "timing", "timings", "transport", "bus",
        "student", "students", "faculty", "exam",
        "exams", "attendance", "course", "courses",
        "btech", "b.tech", "engineering"
    ]

    question_lower = question.lower()

    # 3. Check whether the question is college-related
    if not any(word in question_lower for word in college_keywords):
        return (
            "Sorry! I am CollegeMate AI. 🎓\n\n"
            "I can answer questions about college timings, "
            "branches, admissions, hostel facilities, fees, "
            "placements, scholarships, transportation, and exams.\n\n"
            "Please ask a college-related question."
        )

    # 4. Retrieve relevant information
    results = retrieve_information(question, top_k=3)

    if not results:
        return (
            "Sorry, I could not find this information "
            "in my college database. Please contact "
            "the college administration."
        )

    # 5. Select the best matching information
    best_result = results[0]

    # 6. Check the similarity score
    if best_result["score"] < 0.30:
        return (
            "Sorry, I could not find reliable information "
            "about this question in my college database."
        )

    # 7. Return the answer
    return (
        f"📚 Topic: {best_result['topic']}\n\n"
        f"{best_result['information']}"
    )


print("CollegeMate AI answer function updated successfully!")

CollegeMate AI answer function updated successfully!


In [13]:
# ============================================
# TEST COLLEGEMATE AI
# ============================================

test_questions = [
    "What are the college timings?",
    "Which branches are available?",
    "What documents are required for admission?",
    "Tell me about hostel facilities",
    "What scholarships are available?",
    "How can I cook biryani?",
    "What is the weather today?",
    ""
]

for question in test_questions:

    print("\n" + "=" * 60)
    print("Student Question:", repr(question))
    print("-" * 60)

    answer = generate_answer(question)

    print("CollegeMate AI:")
    print(answer)

    print("=" * 60)


Student Question: 'What are the college timings?'
------------------------------------------------------------
CollegeMate AI:
📚 Topic: College Timings

The illustrative college schedule is Monday to Saturday, from 9 AM to 4 PM. Confirm the actual timetable with the college.

Student Question: 'Which branches are available?'
------------------------------------------------------------
CollegeMate AI:
📚 Topic: Available Branches

Students can ask about CSE, Artificial Intelligence and Data Science, Artificial Intelligence and Machine Learning, and Cyber Security. Confirm currently offered branches with the college.

Student Question: 'What documents are required for admission?'
------------------------------------------------------------
CollegeMate AI:
📚 Topic: Admission Documents

Commonly requested documents include SSC marks memo, diploma or qualifying examination certificates, rank card, transfer certificate, study certificates, caste certificate when applicable, income certificat